In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [2]:
import pandas as pd

df_test, df_train = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv'), pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')

In [3]:
df_train.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [4]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB


In [5]:
X, y = df_train.loc[:, df_train.columns != 'label'].copy(), df_train['label'].copy()

In [6]:
from sklearn.preprocessing import StandardScaler

In [7]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [8]:
class Layer:
  def __init__(self):
    self.training = True

  def forward(self, x):
    raise NotImplementedError

  def baсkward(self, x):
    raise NotImplementedError

  def train(self):
    self.training = True

  def eval(self):
    self.training = False

  def __call__(self, x):
    return self.forward(x)

In [9]:
class ReLU(Layer):
  def __init__(self):
    super().__init__()
    self.input = None

  def forward(self, x):
    self.input = x
    output = x * (x > 0)
    return output

  def backward(self, grad_output):
    grad_input = grad_output * (self.input > 0)
    return grad_input

In [10]:
relu = ReLU()

x_test = np.array([[-2, -1, 0, 1, 2]], dtype=np.float32)
expected_forward = np.array([[0, 0, 0, 1, 2]], dtype=np.float32)

output = relu.forward(x_test)
print(f"Input: {x_test}")
print(f"Output: {output}")
print(f"Expected: {expected_forward}")

assert np.allclose(output, expected_forward), "ReLU forward pass не работает корректно!"

grad_output = np.ones_like(output)
grad_input = relu.backward(grad_output)
expected_backward = np.array([[0, 0, 0, 1, 1]], dtype=np.float32)

print(f"Gradient output: {grad_output}")
print(f"Gradient input: {grad_input}")
print(f"Expected gradient: {expected_backward}")

assert np.allclose(grad_input, expected_backward), "ReLU backward pass не работает корректно!"

Input: [[-2. -1.  0.  1.  2.]]
Output: [[-0. -0.  0.  1.  2.]]
Expected: [[0. 0. 0. 1. 2.]]
Gradient output: [[1. 1. 1. 1. 1.]]
Gradient input: [[0. 0. 0. 1. 1.]]
Expected gradient: [[0. 0. 0. 1. 1.]]


In [11]:
class Sigmoid(Layer):
  def __init__(self):
    super().__init__()
    self.output = None

  def forward(self, x):
    self.output = 1 / (1 + np.exp(-x))
    return self.output

  def backward(self, grad_output):
    return grad_output * self.output * (1 - self.output)

In [12]:
sigmoid = Sigmoid()

x_test = np.array([[-10, -1, 0, 1, 10]], dtype=np.float32)

output = sigmoid.forward(x_test)
print(f"Input: {x_test}")
print(f"Output: {output}")

assert np.all(output > 0) and np.all(output < 1), "Sigmoid должен возвращать значения в диапазоне (0, 1)"

x_sym = np.array([[1]], dtype=np.float32)
out_pos = sigmoid.forward(x_sym)
out_neg = sigmoid.forward(-x_sym)
assert np.allclose(out_neg, 1 - out_pos, atol=1e-6), "Sigmoid должен быть симметричным"

grad_output = np.ones_like(output)
grad_input = sigmoid.backward(grad_output)
print(f"Gradient input: {grad_input}")

assert np.all(grad_input >= 0), "Градиент Sigmoid должен быть неотрицательным"

Input: [[-10.  -1.   0.   1.  10.]]
Output: [[4.539787e-05 2.689414e-01 5.000000e-01 7.310586e-01 9.999546e-01]]
Gradient input: [[0.19661193 0.19661193 0.19661193 0.19661193 0.19661193]]


In [13]:
class Tanh(Layer):
  def __init__(self):
    super().__init__()
    self.output = None

  def forward(self, x):
    self.output = np.tanh(x)
    return self.output

  def backward(self, grad_output):
    return grad_output * (1 - self.output ** 2)

In [14]:
tanh = Tanh()

x_test = np.array([[-10, -1, 0, 1, 10]], dtype=np.float64)

output = tanh.forward(x_test)
print(f"Input: {x_test}")
print(f"Output: {output}")

assert np.all(output > -1) and np.all(output < 1), "Tanh должен возвращать значения в диапазоне (-1, 1)"

x_antisym = np.array([[2]], dtype=np.float32)
out_pos = tanh.forward(x_antisym)
out_neg = tanh.forward(-x_antisym)
assert np.allclose(out_neg, -out_pos, atol=1e-6), "Tanh должен быть антисимметричным"

zero_out = tanh.forward(np.array([[0]], dtype=np.float32))
assert np.allclose(zero_out, 0, atol=1e-6), "tanh(0) должен быть равен 0"

grad_output = np.ones_like(output)
grad_input = tanh.backward(grad_output)
print(f"Gradient input: {grad_input}")

assert np.all(grad_input >= 0), "Градиент Tanh должен быть неотрицательным"

Input: [[-10.  -1.   0.   1.  10.]]
Output: [[-1.         -0.76159416  0.          0.76159416  1.        ]]
Gradient input: [[1. 1. 1. 1. 1.]]


In [15]:
class Linear(Layer):
  def __init__(self, input_size, output_size, bias=True):
    super().__init__()

    self.input_size = input_size
    self.output_size = output_size
    self.use_bias = bias

    if self.use_bias:
      self.bias = np.zeros(output_size)
    else:
      self.bias = None

    std = np.sqrt(2 / input_size)
    self.weight = np.random.randn(input_size, output_size) * std

    self.input = None
    self.grad_weight = None
    self.grad_bias = None

  def forward(self, x):
    self.input = x
    output = x @ self.weight
    if self.use_bias:
      output += self.bias
    return output

  def backward(self, grad_output):
    grad_input = grad_output @ self.weight.T
    self.grad_weight = self.input.T @ grad_output
    self.grad_bias = np.sum(grad_output, axis=0)

    return grad_input

  def update_weights(self, learning_rate=0.01):
    if self.grad_weight is not None:
      self.weight = self.weight - learning_rate * self.grad_weight
    if self.use_bias and self.grad_bias is not None:
      self.bias = self.bias - learning_rate * self.grad_bias

In [16]:
linear = Linear(input_size=3, output_size=2, bias=True)

assert linear.weight.shape == (3, 2), f"Неправильная форма весов: {linear.weight.shape}"
assert linear.bias.shape == (2,), f"Неправильная форма bias: {linear.bias.shape}"

print(f"Веса: \n{linear.weight}")
print(f"Bias: {linear.bias}")

batch_size = 4
x_test = np.random.randn(batch_size, 3).astype(np.float32)

output = linear.forward(x_test)
expected_shape = (batch_size, 2)

print(f"Input shape: {x_test.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected shape: {expected_shape}")

assert output.shape == expected_shape, f"Неправильная форма выхода: {output.shape}"

grad_output = np.random.randn(*output.shape).astype(np.float32)
grad_input = linear.backward(grad_output)

print(f"Gradient input shape: {grad_input.shape}")
print(f"Gradient weight shape: {linear.grad_weight.shape}")
print(f"Gradient bias shape: {linear.grad_bias.shape}")

assert grad_input.shape == x_test.shape, "Неправильная форма градиента по входу"
assert linear.grad_weight.shape == linear.weight.shape, "Неправильная форма градиента по весам"
assert linear.grad_bias.shape == linear.bias.shape, "Неправильная форма градиента по bias"

Веса: 
[[ 0.40556541 -0.11289233]
 [ 0.52883548  1.24354867]
 [-0.19118543 -0.19117202]]
Bias: [0. 0.]
Input shape: (4, 3)
Output shape: (4, 2)
Expected shape: (4, 2)
Gradient input shape: (4, 3)
Gradient weight shape: (3, 2)
Gradient bias shape: (2,)


In [17]:
class Sequential(Layer):
  def __init__(self, *layers):
    super().__init__()
    self.layers = list(layers)
    self.layers_outputs = []

  def add(self, layer):
    self.layers.append(layer)

  def forward(self, x):
    self.layers_outputs.clear()
    output = x
    for layer in self.layers:
      output = layer.forward(output)
      self.layers_outputs.append(output)
    return output

  def backward(self, grad_output):
    grad = grad_output
    for layer in reversed(self.layers):
      grad = layer.backward(grad)
    return grad

  def train(self):
        super().train()
        for layer in self.layers:
            layer.train()

  def eval(self):
      super().eval()
      for layer in self.layers:
          layer.eval()

  def __len__(self):
    return len(self.layers)

  def __getitem__(self, idx):
    return self.layers[idx]

In [18]:
class Dropout(Layer):
  def __init__(self, dropout_rate=0.5):
    super().__init__()
    self.rate = dropout_rate
    self.mask = None

  def forward(self, x):
    if not self.training or self.rate == 0.0:
      output = x
      self.mask = None
    else:
      self.mask = np.random.binomial(1, 1 - self.rate, x.shape)
      output = x * self.mask / (1 - self.rate)
    return output

  def backward(self, grad):
    if not self.training or self.rate == 0.0:
      grad_input = grad
    else:
      grad_input = grad * self.mask / (1 - self.rate)
    return grad_input

In [19]:
dropout = Dropout(dropout_rate=0.5)

x_test = np.ones((100, 10), dtype=np.float32)

dropout.train()
output_train = dropout.forward(x_test)

print(f"Режим обучения:")
print(f"Input mean: {x_test.mean():.3f}")
print(f"Output mean: {output_train.mean():.3f}")
print(f"Proportion of zeros: {(output_train == 0).mean():.3f}")

zeros_ratio = (output_train == 0).mean()
expected_zeros = 0.5  # dropout_rate
assert abs(zeros_ratio - expected_zeros) < 0.1, f"Неправильная доля нулевых значений: {zeros_ratio}"

assert abs(output_train.mean() - x_test.mean()) < 0.1, "Неправильное масштабирование в режиме обучения"

dropout.eval()
output_eval = dropout.forward(x_test)

print(f"\nРежим инференса:")
print(f"Output mean: {output_eval.mean():.3f}")
print(f"Proportion of zeros: {(output_eval == 0).mean():.3f}")

assert np.allclose(output_eval, x_test), "В режиме инференса выход должен совпадать с входом"

dropout.train()
output_train = dropout.forward(x_test)
grad_output = np.ones_like(output_train)
grad_input = dropout.backward(grad_output)

print(f"\nGradient test:")
print(f"Grad input shape: {grad_input.shape}")
print(f"Grad input mean: {grad_input.mean():.3f}")

assert grad_input.shape == x_test.shape, "Неправильная форма градиента"

Режим обучения:
Input mean: 1.000
Output mean: 1.008
Proportion of zeros: 0.496

Режим инференса:
Output mean: 1.000
Proportion of zeros: 0.000

Gradient test:
Grad input shape: (100, 10)
Grad input mean: 1.018


In [20]:
class BatchNorm(Layer):
  def __init__(self, num_features, eps=1e-5, momentum=0.1):
    super().__init__()
    self.num_features = num_features
    self.eps = eps
    self.momentum = momentum

    self.beta, self.gamma = np.zeros(num_features), np.ones(num_features)

    self.running_mean, self.running_var = np.zeros(num_features), np.ones(num_features)

    self.batch_mean = None
    self.batch_var = None

    self.normalized = None
    self.input = None

    self.grad_gamma, self.grad_beta = None, None

  def forward(self, x):
    self.input = x
    if self.training:
      self.batch_mean = x.mean(axis=0)
      self.batch_var = x.var(axis=0)

      self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.batch_mean
      self.running_var = (1 - self.momentum) * self.running_var + self.momentum * self.batch_var

      mean, var = self.batch_mean, self.batch_var
    else:
      mean, var = self.running_mean, self.running_var
    self.normalized = (x - mean) / np.sqrt(var + self.eps)
    output = self.gamma * self.normalized + self.beta
    return output

  def backward(self, grad_output):
    n = self.input.shape[0]
    self.grad_gamma = np.sum(self.normalized * grad_output, axis=0)
    self.grad_beta = np.sum(grad_output, axis=0)

    grad_input = (self.gamma / (n * np.sqrt(self.batch_var + self.eps))) * (
        n * grad_output
        - self.grad_beta
        - self.normalized * self.grad_gamma
    )

    return grad_input

  def update_weights(self, learning_rate=0.01):
        if self.grad_gamma is not None:
            self.gamma -= learning_rate * self.grad_gamma

        if self.grad_beta is not None:
            self.beta -= learning_rate * self.grad_beta

In [21]:
batch_norm = BatchNorm(num_features=4)

assert np.allclose(batch_norm.gamma, 1.0), "Gamma должно инициализироваться единицами"
assert np.allclose(batch_norm.beta, 0.0), "Beta должно инициализироваться нулями"
assert np.allclose(batch_norm.running_mean, 0.0), "Running mean должно инициализироваться нулями"
assert np.allclose(batch_norm.running_var, 1.0), "Running var должно инициализироваться единицами"

print(f"Gamma: {batch_norm.gamma}")
print(f"Beta: {batch_norm.beta}")

x_test = np.array([
    [1, 2, 3, 4],
    [2, 3, 4, 5],
    [3, 4, 5, 6]
], dtype=np.float32)

print(f"Input: \n{x_test}")
print(f"Input mean per feature: {x_test.mean(axis=0)}")
print(f"Input std per feature: {x_test.std(axis=0)}")

batch_norm.train()
output = batch_norm.forward(x_test)

print(f"\nOutput: \n{output}")
print(f"Output mean per feature: {output.mean(axis=0)}")
print(f"Output std per feature: {output.std(axis=0)}")

assert np.allclose(output.mean(axis=0), 0, atol=1e-6), "Среднее должно быть близко к 0"
assert np.allclose(output.std(axis=0), 1, atol=1e-6), "Стандартное отклонение должно быть близко к 1"

print(f"\nRunning mean: {batch_norm.running_mean}")
print(f"Running var: {batch_norm.running_var}")

grad_output = np.ones_like(output)
grad_input = batch_norm.backward(grad_output)

print(f"\nGradient input shape: {grad_input.shape}")
print(f"Gradient gamma shape: {batch_norm.grad_gamma.shape}")
print(f"Gradient beta shape: {batch_norm.grad_beta.shape}")

assert grad_input.shape == x_test.shape, "Неправильная форма градиента по входу"
assert batch_norm.grad_gamma.shape == batch_norm.gamma.shape, "Неправильная форма градиента gamma"
assert batch_norm.grad_beta.shape == batch_norm.beta.shape, "Неправильная форма градиента beta"

batch_norm.eval()
output_eval = batch_norm.forward(x_test)
print(f"\nInference mode output mean: {output_eval.mean(axis=0)}")

Gamma: [1. 1. 1. 1.]
Beta: [0. 0. 0. 0.]
Input: 
[[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]]
Input mean per feature: [2. 3. 4. 5.]
Input std per feature: [0.8164966 0.8164966 0.8164966 0.8164966]

Output: 
[[-1.22473562 -1.22473562 -1.22473562 -1.22473562]
 [ 0.          0.          0.          0.        ]
 [ 1.22473562  1.22473562  1.22473562  1.22473562]]
Output mean per feature: [0. 0. 0. 0.]
Output std per feature: [0.99999244 0.99999244 0.99999244 0.99999244]

Running mean: [0.2        0.30000001 0.40000001 0.5       ]
Running var: [0.96666667 0.96666667 0.96666667 0.96666667]

Gradient input shape: (3, 4)
Gradient gamma shape: (4,)
Gradient beta shape: (4,)

Inference mode output mean: [1.83076198 2.74614297 3.66152397 4.57690497]


In [22]:
class Adam:
  def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    self.lr = learning_rate
    self.b1, self.b2 = beta1, beta2
    self.eps = eps

    self.m, self.v = {}, {}
    self.t = 0

  def update(self, layer, layer_id):
    self.t += 1
    if hasattr(layer, 'grad_weight') and layer.grad_weight is not None:
      key = f"{layer_id}_weight"
      if key not in self.m:
        self.m[key] = np.zeros_like(layer.weight)
        self.v[key] = np.zeros_like(layer.weight)
      self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * layer.grad_weight
      self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * layer.grad_weight ** 2

      m_corr = self.m[key] / (1 - self.b1 ** self.t)
      v_corr = self.v[key] / (1 - self.b2 ** self.t)

      layer.weight -= self.lr * m_corr / (np.sqrt(v_corr) + self.eps)

    if hasattr(layer, 'grad_bias') and layer.grad_bias is not None:
      key = f"{layer_id}_bias"
      if key not in self.m:
        self.m[key] = np.zeros_like(layer.bias)
        self.v[key] = np.zeros_like(layer.bias)
      self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * layer.grad_bias
      self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * layer.grad_bias ** 2

      m_corr = self.m[key] / (1 - self.b1 ** self.t)
      v_corr = self.v[key] / (1 - self.b2 ** self.t)

      layer.bias -= self.lr * m_corr / (np.sqrt(v_corr) + self.eps)

    if hasattr(layer, 'grad_gamma') and layer.grad_gamma is not None:
      key = f"{layer_id}_gamma"
      if key not in self.m:
        self.m[key] = np.zeros_like(layer.gamma)
        self.v[key] = np.zeros_like(layer.gamma)
      self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * layer.grad_gamma
      self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * layer.grad_gamma ** 2

      m_corr = self.m[key] / (1 - self.b1 ** self.t)
      v_corr = self.v[key] / (1 - self.b2 ** self.t)

      layer.gamma -= self.lr * m_corr / (np.sqrt(v_corr) + self.eps)

    if hasattr(layer, 'grad_beta') and layer.grad_beta is not None:
      key = f"{layer_id}_beta"
      if key not in self.m:
        self.m[key] = np.zeros_like(layer.beta)
        self.v[key] = np.zeros_like(layer.beta)
      self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * layer.grad_beta
      self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * layer.grad_beta ** 2

      m_corr = self.m[key] / (1 - self.b1 ** self.t)
      v_corr = self.v[key] / (1 - self.b2 ** self.t)

      layer.beta -= self.lr * m_corr / (np.sqrt(v_corr) + self.eps)

  def zero_grad(self, layers):
    for layer in layers:
      if hasattr(layer, 'grad_weight'):
                layer.grad_weight = None
      if hasattr(layer, 'grad_bias'):
          layer.grad_bias = None
      if hasattr(layer, 'grad_gamma'):
          layer.grad_gamma = None
      if hasattr(layer, 'grad_beta'):
          layer.grad_beta = None

In [23]:
layer = Linear(3, 2)
adam = Adam(learning_rate=0.01)

layer.grad_weight = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]], dtype=np.float32)
layer.grad_bias = np.array([0.1, 0.2], dtype=np.float32)

initial_weight = layer.weight.copy()
initial_bias = layer.bias.copy()

print(f"Initial weight: \n{initial_weight}")
print(f"Initial bias: {initial_bias}")
print(f"Weight gradient: \n{layer.grad_weight}")
print(f"Bias gradient: {layer.grad_bias}")

assert len(adam.m) == 0, "Моменты должны быть пустыми при инициализации"
assert len(adam.v) == 0, "Моменты должны быть пустыми при инициализации"
assert adam.t == 0, "Time step должен быть равен 0"

adam.update(layer, "test_layer")

print(f"\nAfter 1 step:")
print(f"Updated weight: \n{layer.weight}")
print(f"Updated bias: {layer.bias}")
print(f"Time step: {adam.t}")

assert not np.allclose(layer.weight, initial_weight), "Веса должны измениться после обновления"
assert not np.allclose(layer.bias, initial_bias), "Bias должен измениться после обновления"

assert "test_layer_weight" in adam.m, "Момент для весов должен быть создан"
assert "test_layer_bias" in adam.m, "Момент для bias должен быть создан"
assert "test_layer_weight" in adam.v, "Момент для весов должен быть создан"
assert "test_layer_bias" in adam.v, "Момент для bias должен быть создан"

assert adam.m["test_layer_weight"].shape == layer.weight.shape, "Неправильная форма момента весов"
assert adam.m["test_layer_bias"].shape == layer.bias.shape, "Неправильная форма момента bias"

adam.zero_grad([layer])
assert layer.grad_weight is None, "Градиенты весов должны быть обнулены"
assert layer.grad_bias is None, "Градиенты bias должны быть обнулены"

Initial weight: 
[[ 0.23968905 -1.10764441]
 [ 0.38083849 -0.02910115]
 [-1.31874961  0.9510057 ]]
Initial bias: [0. 0.]
Weight gradient: 
[[0.1 0.2]
 [0.3 0.4]
 [0.5 0.6]]
Bias gradient: [0.1 0.2]

After 1 step:
Updated weight: 
[[ 0.22968905 -1.1176444 ]
 [ 0.37083849 -0.03910115]
 [-1.32874961  0.9410057 ]]
Updated bias: [-0.01 -0.01]
Time step: 1


In [24]:
class CrossEntropyLoss:
    def __init__(self):
        self.predictions = None
        self.targets = None

    def forward(self, predictions, targets):
        self.predictions = predictions
        self.targets = targets

        pred_max = np.max(predictions, axis=-1, keepdims=True)
        softmax_pred = predictions - np.log(np.sum(np.exp(predictions - pred_max), axis=-1, keepdims=True)) - pred_max
        loss = -softmax_pred[np.arange(targets.shape[0]), targets].mean()

        return loss

    def backward(self):
        grad = (softmax(self.predictions) - one_hot_encode(self.targets, self.predictions.shape[1])) / self.predictions.shape[0]

        return grad


class MSELoss:
    def __init__(self):
        self.predictions = None
        self.targets = None

    def forward(self, predictions, targets):
        self.predictions = predictions
        self.targets = targets

        loss = np.mean((targets - predictions) ** 2)

        return loss

    def backward(self):
        grad = 2 * (self.predictions - self.targets) / self.predictions.size

        return grad


def softmax(x):
    x_max = np.max(x, axis=-1, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def one_hot_encode(labels, num_classes):
    n = len(labels)
    one_hot_array = np.zeros((n, num_classes))
    one_hot_array[np.arange(n), labels] = 1.0
    return one_hot_array


In [25]:
ce_loss = CrossEntropyLoss()

predictions = np.array([[2.0, 1.0, 0.1], [1.0, 3.0, 0.2]], dtype=np.float32)
targets = np.array([0, 1], dtype=np.int32)

print(f"Predictions: \n{predictions}")
print(f"Targets: {targets}")

loss_value = ce_loss.forward(predictions, targets)
print(f"CrossEntropy Loss: {loss_value:.4f}")

assert loss_value > 0, "CrossEntropy loss должен быть положительным"

grad = ce_loss.backward()
print(f"Gradient shape: {grad.shape}")
print(f"Gradient: \n{grad}")

assert grad.shape == predictions.shape, "Неправильная форма градиента CrossEntropy"

assert np.allclose(grad.sum(axis=1), 0, atol=1e-6), "Сумма градиентов по классам должна быть 0"

mse_loss = MSELoss()

predictions_reg = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
targets_reg = np.array([[1.5, 2.5], [2.5, 3.5]], dtype=np.float32)

print(f"Predictions: \n{predictions_reg}")
print(f"Targets: \n{targets_reg}")

mse_value = mse_loss.forward(predictions_reg, targets_reg)
print(f"MSE Loss: {mse_value:.4f}")

assert mse_value >= 0, "MSE loss должен быть неотрицательным"

grad_mse = mse_loss.backward()
print(f"MSE Gradient shape: {grad_mse.shape}")
print(f"MSE Gradient: \n{grad_mse}")

assert grad_mse.shape == predictions_reg.shape, "Неправильная форма градиента MSE"

x_softmax = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]], dtype=np.float32)
softmax_output = softmax(x_softmax)

print(f"Input: \n{x_softmax}")
print(f"Softmax output: \n{softmax_output}")

assert np.allclose(softmax_output.sum(axis=1), 1.0), "Сумма softmax должна быть равна 1"

assert np.all(softmax_output > 0), "Все значения softmax должны быть положительными"
assert np.all(softmax_output < 1), "Все значения softmax должны быть меньше 1"

labels = np.array([0, 2, 1, 0])
one_hot = one_hot_encode(labels, num_classes=3)

print(f"Labels: {labels}")
print(f"One-hot: \n{one_hot}")

assert one_hot.shape == (4, 3), "Неправильная форма one-hot кодировки"

assert np.all(one_hot.sum(axis=1) == 1), "Каждая строка должна содержать ровно одну единицу"

Predictions: 
[[2.  1.  0.1]
 [1.  3.  0.2]]
Targets: [0 1]
CrossEntropy Loss: 0.2981
Gradient shape: (2, 3)
Gradient: 
[[-0.17049944  0.12121648  0.04928295]
 [ 0.05657142 -0.08199063  0.02541918]]
Predictions: 
[[1. 2.]
 [3. 4.]]
Targets: 
[[1.5 2.5]
 [2.5 3.5]]
MSE Loss: 0.2500
MSE Gradient shape: (2, 2)
MSE Gradient: 
[[-0.25 -0.25]
 [ 0.25  0.25]]
Input: 
[[1. 2. 3.]
 [1. 1. 1.]]
Softmax output: 
[[0.09003057 0.24472846 0.66524094]
 [0.33333334 0.33333334 0.33333334]]
Labels: [0 2 1 0]
One-hot: 
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


In [26]:
class NeuralNetwork:
    def __init__(self):
        self.model = Sequential(
            Linear(784, 512),
            BatchNorm(512),
            ReLU(),
            Dropout(),
            Linear(512, 256),
            BatchNorm(256),
            ReLU(),
            Dropout(0.3),
            Linear(256, 128),
            BatchNorm(128),
            ReLU(),
            Dropout(0.2),
            Linear(128, 10),
        )
    
    def forward(self, x):
        return self.model.forward(x)
    
    def backward(self, grad_output):
        return self.model.backward(grad_output)
    
    def train(self):
        self.model.train()
    
    def eval(self):
        self.model.eval()
    
    def get_trainable_layers(self):
        trainable_layers = []
        for layer in self.model.layers:
            if hasattr(layer, 'update_weights'):
                trainable_layers.append(layer)
        return trainable_layers

In [32]:
nn, optim, loss = NeuralNetwork(), Adam(), CrossEntropyLoss()

In [28]:
scaler = StandardScaler()

In [29]:
X_np, y_np = scaler.fit_transform(X.values), y.values

In [33]:
nn.train()
n = len(y_np)
train_layers = nn.get_trainable_layers()
for epoch in range(20):
    avg = 0
    count = 0
    sum_corr_val = 0
    indices = np.random.permutation(len(X_np))
    for i in range(0, len(X_np), 32):
        idx = indices[i:i + 32]
        Xb, yb = X_np[idx], y_np[idx]
        pred = nn.forward(Xb)
        sum_corr_val += np.sum((yb == np.argmax(pred, axis=1)), axis=-1)
        loss_val = loss.forward(pred, yb)
        avg += loss_val
        count += 1
        nn.backward(loss.backward())
        for j, t_layer in enumerate(train_layers):
            optim.update(t_layer, j)
        optim.zero_grad(train_layers)
    avg /= count
    print(sum_corr_val / n)
    print(avg)

0.8690476190476191
0.419524944574487
0.9245476190476191
0.24569612539865293
0.9372857142857143
0.20109905262491354
0.9448095238095238
0.17663865417898847
0.9498809523809524
0.15943120588287596
0.9557380952380953
0.1434497216413672
0.9552857142857143
0.13962230422369956
0.9598809523809524
0.12494451508487096
0.9635952380952381
0.11641330724818935
0.9636190476190476
0.11404951280144889
0.9675
0.10079168301463375
0.9682142857142857
0.0986995116912265
0.969452380952381
0.09308353011965324
0.9702142857142857
0.09182244920962156
0.9718571428571429
0.08641376214339452
0.9728809523809524
0.08349957035576112
0.9740238095238095
0.07943978249350897
0.9754285714285714
0.0766608906969312
0.9755
0.07471163441342603
0.977
0.07061057615009565


In [34]:
X_test = scaler.transform(df_test.values)

In [35]:
nn.eval()
res = nn.forward(X_test)
test_pred = np.argmax(res, axis=1)

In [36]:
sample = pd.read_csv('/kaggle/input/competitions/digit-recognizer/sample_submission.csv')
sample['Label'] = test_pred
sample.to_csv('submission.csv', index=False)